In [1]:
import os, urllib.request
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from numpy.lib.stride_tricks import sliding_window_view

# ============================================================
# 1. Data: Tiny Shakespeare, character level
# ============================================================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", "r", encoding="utf-8").read()

chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
encoded = np.array([char2idx[c] for c in text], dtype=np.int32)
print("characters:", len(text), "| vocab:", vocab_size)

# ============================================================
# 2. Hyperparameters
# ============================================================
SEQ_LEN = 64          # the flatten layer has (SEQ_LEN*EMBED_DIM)^2 weights, so keep these small
EMBED_DIM = 32
FF_DIM = 128
NUM_LAYERS = 3
CONV_K = 4            # the conv "reader" looks at 4 characters at a time
DROPOUT = 0.1
BATCH_SIZE = 64
EPOCHS = 3
STEPS_PER_EPOCH = 500

# ============================================================
# 3. Train / validation split (last 10% held out)
# ============================================================
n_val = int(0.1 * len(encoded))
train_ids, val_ids = encoded[:-n_val], encoded[-n_val:]

def make_ds(data, training):
    w = sliding_window_view(data, SEQ_LEN + 1)
    X, y = w[:, :-1].copy(), w[:, 1:].copy()
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(10000).repeat()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds = make_ds(train_ids, True), make_ds(val_ids, False)

# ============================================================
# 4. Layers
# ============================================================
class RMSNorm(layers.Layer):
    def build(self, shape):
        self.w = self.add_weight(shape=(shape[-1],), initializer="ones", name="w")
    def call(self, x):
        return x * tf.math.rsqrt(tf.reduce_mean(tf.square(x), -1, keepdims=True) + 1e-6) * self.w

class PosEmb(layers.Layer):
    def __init__(self, T, d, **kw):
        super().__init__(**kw)
        self.emb = layers.Embedding(T, d)
    def call(self, x):
        return x + self.emb(tf.range(tf.shape(x)[1]))

class MaskedDense(layers.Layer):
    """Dense layer on the FLATTENED sequence. Every weight that would let
    position t read a later token is multiplied by 0 (the causal mask)."""
    def __init__(self, T, d, **kw):
        super().__init__(**kw)
        self.T, self.d = T, d
        pos = np.repeat(np.arange(T), d)                         # token index of each flat slot
        self.mask_np = (pos[:, None] <= pos[None, :]).astype("float32")  # [flat_in, flat_out]
    def build(self, shape):
        n = self.T * self.d
        self.w = self.add_weight(shape=(n, n), name="w",
                                 initializer=keras.initializers.RandomNormal(stddev=0.01))
        self.b = self.add_weight(shape=(n,), initializer="zeros", name="b")
    def call(self, x):
        return tf.matmul(x, self.w * tf.constant(self.mask_np)) + self.b

def block(x):
    # (a) conv: the local "reader" (causal = padded on the left only)
    c = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal", activation="silu")(RMSNorm()(x))
    x = layers.Add()([x, c])
    # (b) flatten -> causally masked dense -> back to (tokens, dim)
    m = layers.Flatten()(RMSNorm()(x))
    m = MaskedDense(SEQ_LEN, EMBED_DIM)(m)
    m = layers.Reshape((SEQ_LEN, EMBED_DIM))(m)
    x = layers.Add()([x, m])
    # (c) ffn
    f = layers.Dense(FF_DIM, activation="silu")(RMSNorm()(x))
    f = layers.Dropout(DROPOUT)(layers.Dense(EMBED_DIM)(f))
    return layers.Add()([x, f])

# ============================================================
# 5. Model: input, embedding, pos, rmsnorm, conv, flatten, causal mask, ffn, dense
# ============================================================
I = layers.Input(shape=(SEQ_LEN,), dtype="int32")
x = layers.Embedding(vocab_size, EMBED_DIM)(I)
x = PosEmb(SEQ_LEN, EMBED_DIM)(x)
for _ in range(NUM_LAYERS):
    x = block(x)
out = layers.Dense(vocab_size)(RMSNorm()(x))     # raw logits, no softmax (loss uses from_logits=True)
model = keras.Model(I, out)
model.summary()

# ============================================================
# 6. LEAK TEST: run BEFORE training
# ============================================================
a = np.random.randint(0, vocab_size, (1, SEQ_LEN)); b = a.copy()
b[0, 40] = (b[0, 40] + 1) % vocab_size           # change token 40
diff = np.abs(model(a, training=False).numpy()[0, :40] - model(b, training=False).numpy()[0, :40]).max()
print("leak test (positions before 40 must not change):", diff)
assert diff < 1e-4, "FUTURE LEAK, do not train"

# ============================================================
# 7. Train
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_ds, epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
                    validation_data=val_ds, validation_steps=200)

# ============================================================
# 8. Text generation
# ============================================================
def generate_text(model, start_string, length=500, temperature=1.0):
    ids = [char2idx[c] for c in start_string]
    out = start_string
    for _ in range(length):
        x = ids[-SEQ_LEN:]
        n_real = len(x)
        x = [0] * (SEQ_LEN - n_real) + x                 # left-pad (safe: the model is causal)
        logits = model(np.array([x]), training=False)[:, -1, :] / temperature
        nxt = int(tf.random.categorical(logits, num_samples=1)[0, 0])
        out += idx2char[nxt]; ids.append(nxt)
    return out

for name, temp in [("ROMEO:", 0.8), ("KING:", 1.0), ("JULIET:", 1.2)]:
    print("\n" + "=" * 70, f"\nSTART {name!r}  temperature {temp}\n" + "=" * 70)
    print(generate_text(model, name, 500, temp))

characters: 1115394 | vocab: 65


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 64, 32)    │      2,080 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_emb (PosEmb)    │ (None, 64, 32)    │      2,048 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm (RMSNorm)  │ (None, 64, 32)    │         32 │ pos_emb[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 64, 32)    │      4,128 │ rms_norm[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 64, 32)    │          0 │ pos_emb[0][0],    │
│                     │                   │            │ conv1d[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_1          │ (None, 64, 32)    │         32 │ add[0][0]         │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 2048)      │          0 │ rms_norm_1[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masked_dense        │ (None, 2048)      │  4,196,352 │ flatten[0][0]     │
│ (MaskedDense)       │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 64, 32)    │          0 │ masked_dense[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 64, 32)    │          0 │ add[0][0],        │
│                     │                   │            │ reshape[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_2          │ (None, 64, 32)    │         32 │ add_1[0][0]       │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64, 128)   │      4,224 │ rms_norm_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 64, 32)    │      4,128 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64, 32)    │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 64, 32)    │          0 │ add_1[0][0],      │
│                     │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_3          │ (None, 64, 32)    │         32 │ add_2[0][0]       │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 64, 32)    │      4,128 │ rms_norm_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 64, 32)    │          0 │ add_2[0][0],      │
│                     │                   │            │ conv1d_1[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_4          │ (None, 64, 32)    │         32 │ add_3[0][0]     

 Total params: 12,633,089 (48.19 MB)

 Trainable params: 12,633,089 (48.19 MB)

 Non-trainable params: 0 (0.00 B)

leak test (positions before 40 must not change): 0.0
Epoch 1/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 28s 18ms/step - accuracy: 0.3210 - loss: 2.6132 - val_accuracy: 0.3391 - val_loss: 2.4902
Epoch 2/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.4372 - loss: 1.9951 - val_accuracy: 0.3619 - val_loss: 2.4161
Epoch 3/3
500/500 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4821 - loss: 1.7888 - val_accuracy: 0.3793 - val_loss: 2.3722

START 'ROMEO:'  temperature 0.8
ROMEO: he so upe?

VoXUfuI.

?

MENENIUS:
I's licbunis made.

SICINIUS:
So! the go you.

MORUSUCNIL:
Wowh, wiy with theme here god the sother mume me dive at have sower of titheries would that in you of would baride for cammon that the hearcouses you. I seall powelcherave weol, I am me to our and he hish afai their do.

CORIOLANUS:
Sad you' a'll he mir, we han me people not if day infarces, whid
The hows of will goted:
What the monozes's ander bools woll, and of be ave stien fore andist you prame me s

START 'KING:'  tempera

In [2]:
import os, urllib.request
import numpy as np
import tensorflow as tf
import keras
from keras import layers
from numpy.lib.stride_tricks import sliding_window_view

# ============================================================
# 1. Data: Tiny Shakespeare, character level
# ============================================================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", "r", encoding="utf-8").read()
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
encoded = np.array([char2idx[c] for c in text], dtype=np.int32)
print("characters:", len(text), "| vocab:", vocab_size)

# ============================================================
# 2. Hyperparameters
# ============================================================
SEQ_LEN = 128
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 512
NUM_LAYERS = 4
CONV_K = 4            # kernel of the Q/K/V convs. CONV_K = 1 is the plain-Dense baseline (same model otherwise)
DROPOUT = 0.1
BATCH_SIZE = 64
EPOCHS = 10
STEPS_PER_EPOCH = 1000

n_val = int(0.1 * len(encoded))
train_ids, val_ids = encoded[:-n_val], encoded[-n_val:]

def make_ds(data, training):
    w = sliding_window_view(data, SEQ_LEN + 1)
    X, y = w[:, :-1].copy(), w[:, 1:].copy()
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(10000).repeat()
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds = make_ds(train_ids, True), make_ds(val_ids, False)

# ============================================================
# 3. Layers
# ============================================================
class RMSNorm(layers.Layer):
    def build(self, shape):
        self.w = self.add_weight(shape=(shape[-1],), initializer="ones", name="w")
    def call(self, x):
        return x * tf.math.rsqrt(tf.reduce_mean(tf.square(x), -1, keepdims=True) + 1e-6) * self.w

class PosEmb(layers.Layer):
    def __init__(self, T, d, **kw):
        super().__init__(**kw)
        self.emb = layers.Embedding(T, d)
    def call(self, x):
        return x + self.emb(tf.range(tf.shape(x)[1]))

class CausalLinearAttention(layers.Layer):
    """Takes already-computed Q, K, V (from the convs) and applies
       out_t = phi(q_t) . [ sum_{s<=t} phi(k_s) v_s^T ]  /  phi(q_t) . sum_{s<=t} phi(k_s)
       i.e. Q * (K^T * V) with a running sum, so position t never sees the future."""
    def __init__(self, T, heads, **kw):
        super().__init__(**kw)
        self.T, self.h = T, heads
    def call(self, inputs):
        q, k, v = inputs
        B = tf.shape(q)[0]
        d = q.shape[-1]
        dh = d // self.h
        sp = lambda t: tf.reshape(t, (B, self.T, self.h, dh))
        q = tf.nn.elu(sp(q)) + 1.0
        k = tf.nn.elu(sp(k)) + 1.0
        v = sp(v)
        S = tf.cumsum(tf.einsum('bthd,bthe->bthde', k, v), axis=1)   # running K^T V
        z = tf.cumsum(k, axis=1)                                      # running normalizer
        num = tf.einsum('bthd,bthde->bthe', q, S)
        den = tf.einsum('bthd,bthd->bth', q, z)[..., None] + 1e-6
        return tf.reshape(num / den, (B, self.T, d))

def block(x):
    h = RMSNorm()(x)
    # Q, K, V each come from their own causal conv (padding="causal" = left padding only)
    q = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal")(h)
    k = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal")(h)
    v = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal")(h)
    a = CausalLinearAttention(SEQ_LEN, NUM_HEADS)([q, k, v])
    a = layers.Dense(EMBED_DIM)(a)                                    # output projection
    x = layers.Add()([x, a])
    f = layers.Dense(FF_DIM, activation="silu")(RMSNorm()(x))         # ffn
    f = layers.Dropout(DROPOUT)(layers.Dense(EMBED_DIM)(f))
    return layers.Add()([x, f])

# ============================================================
# 4. Model: input, embedding, pos, [rmsnorm, conv->Q,K,V, linear attention, ffn] x N, dense
# ============================================================
I = layers.Input(shape=(SEQ_LEN,), dtype="int32")
x = layers.Embedding(vocab_size, EMBED_DIM)(I)
x = PosEmb(SEQ_LEN, EMBED_DIM)(x)
for _ in range(NUM_LAYERS):
    x = block(x)
out = layers.Dense(vocab_size)(RMSNorm()(x))      # logits, no softmax
model = keras.Model(I, out)
model.summary()

# ============================================================
# 5. LEAK TEST (run BEFORE training)
# ============================================================
a = np.random.randint(0, vocab_size, (1, SEQ_LEN)); b = a.copy()
b[0, 100] = (b[0, 100] + 1) % vocab_size
diff = np.abs(model(a, training=False).numpy()[0, :100] - model(b, training=False).numpy()[0, :100]).max()
print("leak test (positions before 100 must not change):", diff)
assert diff < 1e-4, "FUTURE LEAK, do not train"

# ============================================================
# 6. Train
# ============================================================
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(train_ds, epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
                    validation_data=val_ds, validation_steps=200)
print(f"\nFINAL  CONV_K={CONV_K}  params={model.count_params():,}  val_loss={history.history['val_loss'][-1]:.4f}")

# ============================================================
# 7. Text generation
# ============================================================
def generate_text(model, start_string, length=500, temperature=1.0):
    ids = [char2idx[c] for c in start_string]
    out = start_string
    for _ in range(length):
        x = ids[-SEQ_LEN:]
        x = [0] * (SEQ_LEN - len(x)) + x
        logits = model(np.array([x]), training=False)[:, -1, :] / temperature
        nxt = int(tf.random.categorical(logits, num_samples=1)[0, 0])
        out += idx2char[nxt]; ids.append(nxt)
    return out

for name, temp in [("ROMEO:", 0.8), ("KING:", 1.0), ("JULIET:", 1.2)]:
    print("\n" + "=" * 70, f"\nSTART {name!r}  temperature {temp}\n" + "=" * 70)
    print(generate_text(model, name, 500, temp))

characters: 1115394 | vocab: 65


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 128, 128)  │      8,320 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_emb_1 (PosEmb)  │ (None, 128, 128)  │     16,384 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_10         │ (None, 128, 128)  │        128 │ pos_emb_1[0][0]   │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 128, 128)  │     65,664 │ rms_norm_10[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 128, 128)  │     65,664 │ rms_norm_10[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 128, 128)  │     65,664 │ rms_norm_10[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ causal_linear_atte… │ (None, 128, 128)  │          0 │ conv1d_3[0][0],   │
│ (CausalLinearAtten… │                   │            │ conv1d_4[0][0],   │
│                     │                   │            │ conv1d_5[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 128, 128)  │     16,512 │ causal_linear_at… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 128, 128)  │          0 │ pos_emb_1[0][0],  │
│                     │                   │            │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_11         │ (None, 128, 128)  │        128 │ add_9[0][0]       │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128, 512)  │     66,048 │ rms_norm_11[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 128, 128)  │     65,664 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128, 128)  │          0 │ dense_9[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 128, 128)  │          0 │ add_9[0][0],      │
│                     │                   │            │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_12         │ (None, 128, 128)  │        128 │ add_10[0][0]      │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 128, 128)  │     65,664 │ rms_norm_12[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_7 (Conv1D)   │ (None, 128, 128)  │     65,664 │ rms_norm_12[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_8 (Conv1D)   │ (None, 128, 128)  │     65,664 │ rms_norm_12[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ causal_linear_atte… │ (None, 128, 128)  │          0 │ conv1d_6[0][0],   │
│ (CausalLinearAtten… │                   │            │ conv1d_7[0][0], 

 Total params: 1,415,105 (5.40 MB)

 Trainable params: 1,415,105 (5.40 MB)

 Non-trainable params: 0 (0.00 B)

leak test (positions before 100 must not change): 0.0
Epoch 1/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 103s 79ms/step - accuracy: 0.4861 - loss: 1.8123 - val_accuracy: 0.3805 - val_loss: 2.6131
Epoch 2/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 79s 79ms/step - accuracy: 0.6985 - loss: 1.0133 - val_accuracy: 0.3883 - val_loss: 2.8107
Epoch 3/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 82s 82ms/step - accuracy: 0.7215 - loss: 0.9492 - val_accuracy: 0.4013 - val_loss: 2.8432
Epoch 4/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 84s 84ms/step - accuracy: 0.7587 - loss: 0.8223 - val_accuracy: 0.4049 - val_loss: 2.8533
Epoch 5/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 85s 85ms/step - accuracy: 0.7733 - loss: 0.7803 - val_accuracy: 0.3945 - val_loss: 2.9717
Epoch 6/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 86s 86ms/step - accuracy: 0.7748 - loss: 0.7743 - val_accuracy: 0.4029 - val_loss: 3.0132
Epoch 7/10
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 85s 85ms/step - accuracy: 0.7848 - loss: 0.7439 - val_accuracy: 0.4130 - val_loss: 2.8795
Epoch 8/10
1000/1000 

In [3]:
import os, urllib.request
import numpy as np
import tensorflow as tf
import keras
from keras import layers

# ============================================================
# 1. Data: Tiny Shakespeare, character level
# ============================================================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", "r", encoding="utf-8").read()
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = np.array(chars)
encoded = np.array([char2idx[c] for c in text], dtype=np.int32)
print("characters:", len(text), "| vocab:", vocab_size)

# ============================================================
# 2. Hyperparameters
# ============================================================
SEQ_LEN = 128
EMBED_DIM = 128
NUM_HEADS = 4
FF_DIM = 512
NUM_LAYERS = 4
CONV_K = 4            # kernel of the Q/K/V convs. CONV_K = 1 is the plain-Dense baseline (same model otherwise)
DROPOUT = 0.1
BATCH_SIZE = 64
EPOCHS = 30             # early stopping will end it sooner
STEPS_PER_EPOCH = 300

n_val = int(0.1 * len(encoded))          # same split as the earlier Shakespeare runs: last 10% held out
train_ids, val_ids = encoded[:-n_val], encoded[-n_val:]

def make_ds(data, training):
    """Windows are picked from GLOBALLY shuffled start positions (shuffle buffer = whole text).
       (The old .shuffle(10000) over ordered stride-1 windows only mixed ~10k neighbouring characters,
        so training walked through the file region by region and memorized each region.)"""
    data_tf = tf.constant(data, dtype=tf.int32)
    n = len(data) - SEQ_LEN - 1
    def gather(ix):
        pos = ix[:, None] + tf.range(SEQ_LEN + 1)[None, :]
        seq = tf.gather(data_tf, pos)
        return seq[:, :-1], seq[:, 1:]
    if training:
        starts = tf.data.Dataset.from_tensor_slices(np.arange(n, dtype=np.int32))
        starts = starts.shuffle(n, reshuffle_each_iteration=True).repeat()
    else:
        starts = tf.data.Dataset.from_tensor_slices(np.arange(0, n, SEQ_LEN, dtype=np.int32))   # non-overlapping, fixed
    return starts.batch(BATCH_SIZE).map(gather, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)

train_ds, val_ds = make_ds(train_ids, True), make_ds(val_ids, False)

# ============================================================
# 3. Layers
# ============================================================
class RMSNorm(layers.Layer):
    def build(self, shape):
        self.w = self.add_weight(shape=(shape[-1],), initializer="ones", name="w")
    def call(self, x):
        return x * tf.math.rsqrt(tf.reduce_mean(tf.square(x), -1, keepdims=True) + 1e-6) * self.w

class PosEmb(layers.Layer):
    def __init__(self, T, d, **kw):
        super().__init__(**kw)
        self.emb = layers.Embedding(T, d)
    def call(self, x):
        return x + self.emb(tf.range(tf.shape(x)[1]))

class CausalLinearAttention(layers.Layer):
    """Takes already-computed Q, K, V (from the convs) and applies
       out_t = phi(q_t) . [ sum_{s<=t} phi(k_s) v_s^T ]  /  phi(q_t) . sum_{s<=t} phi(k_s)
       i.e. Q * (K^T * V) with a running sum, so position t never sees the future."""
    def __init__(self, T, heads, **kw):
        super().__init__(**kw)
        self.T, self.h = T, heads
    def call(self, inputs):
        q, k, v = inputs
        B = tf.shape(q)[0]
        d = q.shape[-1]
        dh = d // self.h
        sp = lambda t: tf.reshape(t, (B, self.T, self.h, dh))
        q = tf.nn.elu(sp(q)) + 1.0
        k = tf.nn.elu(sp(k)) + 1.0
        v = sp(v)
        S = tf.cumsum(tf.einsum('bthd,bthe->bthde', k, v), axis=1)   # running K^T V
        z = tf.cumsum(k, axis=1)                                      # running normalizer
        num = tf.einsum('bthd,bthde->bthe', q, S)
        den = tf.einsum('bthd,bthd->bth', q, z)[..., None] + 1e-6
        return tf.reshape(num / den, (B, self.T, d))

def block(x):
    h = RMSNorm()(x)
    # Q, K, V each come from their own causal conv (padding="causal" = left padding only)
    q = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal")(h)
    k = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal")(h)
    v = layers.Conv1D(EMBED_DIM, CONV_K, padding="causal")(h)
    a = CausalLinearAttention(SEQ_LEN, NUM_HEADS)([q, k, v])
    a = layers.Dense(EMBED_DIM)(a)                                    # output projection
    x = layers.Add()([x, a])
    f = layers.Dense(FF_DIM, activation="silu")(RMSNorm()(x))         # ffn
    f = layers.Dropout(DROPOUT)(layers.Dense(EMBED_DIM)(f))
    return layers.Add()([x, f])

# ============================================================
# 4. Model: input, embedding, pos, [rmsnorm, conv->Q,K,V, linear attention, ffn] x N, dense
# ============================================================
I = layers.Input(shape=(SEQ_LEN,), dtype="int32")
x = layers.Embedding(vocab_size, EMBED_DIM)(I)
x = PosEmb(SEQ_LEN, EMBED_DIM)(x)
for _ in range(NUM_LAYERS):
    x = block(x)
out = layers.Dense(vocab_size)(RMSNorm()(x))      # logits, no softmax
model = keras.Model(I, out)
model.summary()

# ============================================================
# 5. LEAK TEST (run BEFORE training)
# ============================================================
a = np.random.randint(0, vocab_size, (1, SEQ_LEN)); b = a.copy()
b[0, 100] = (b[0, 100] + 1) % vocab_size
diff = np.abs(model(a, training=False).numpy()[0, :100] - model(b, training=False).numpy()[0, :100]).max()
print("leak test (positions before 100 must not change):", diff)
assert diff < 1e-4, "FUTURE LEAK, do not train"

# ============================================================
# 6. Train (same optimizer recipe as the earlier Shakespeare runs, with early stopping on val loss)
# ============================================================
model.compile(
    optimizer=keras.optimizers.AdamW(learning_rate=1e-3, weight_decay=0.01, clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
history = model.fit(
    train_ds, epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH, validation_data=val_ds,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1, min_lr=1e-5)],
)
best = min(history.history["val_loss"])
print(f"\nFINAL  CONV_K={CONV_K}  params={model.count_params():,}  best val_loss={best:.4f}  "
      f"(earlier 462K-param runs on this same split: linear 1.634, softmax 1.622)")

# ============================================================
# 7. Text generation
# ============================================================
def generate_text(model, start_string, length=500, temperature=1.0):
    primer = text[5000:5000 + SEQ_LEN]                      # real Shakespeare fills the window first
    ids = [char2idx[c] for c in (primer + start_string)]
    out = start_string
    for _ in range(length):
        x = ids[-SEQ_LEN:]
        logits = model(np.array([x]), training=False)[:, -1, :] / temperature
        nxt = int(tf.random.categorical(logits, num_samples=1)[0, 0])
        out += idx2char[nxt]; ids.append(nxt)
    return out

for name, temp in [("ROMEO:", 0.8), ("KING:", 1.0), ("JULIET:", 1.2)]:
    print("\n" + "=" * 70, f"\nSTART {name!r}  temperature {temp}\n" + "=" * 70)
    print(generate_text(model, name, 500, temp))

characters: 1115394 | vocab: 65


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 128)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_4         │ (None, 128, 128)  │      8,320 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pos_emb_2 (PosEmb)  │ (None, 128, 128)  │     16,384 │ embedding_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_19         │ (None, 128, 128)  │        128 │ pos_emb_2[0][0]   │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 128, 128)  │     65,664 │ rms_norm_19[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 128, 128)  │     65,664 │ rms_norm_19[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 128, 128)  │     65,664 │ rms_norm_19[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ causal_linear_atte… │ (None, 128, 128)  │          0 │ conv1d_15[0][0],  │
│ (CausalLinearAtten… │                   │            │ conv1d_16[0][0],  │
│                     │                   │            │ conv1d_17[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 128, 128)  │     16,512 │ causal_linear_at… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_17 (Add)        │ (None, 128, 128)  │          0 │ pos_emb_2[0][0],  │
│                     │                   │            │ dense_20[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_20         │ (None, 128, 128)  │        128 │ add_17[0][0]      │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 128, 512)  │     66,048 │ rms_norm_20[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 128, 128)  │     65,664 │ dense_21[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 128, 128)  │          0 │ dense_22[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_18 (Add)        │ (None, 128, 128)  │          0 │ add_17[0][0],     │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rms_norm_21         │ (None, 128, 128)  │        128 │ add_18[0][0]      │
│ (RMSNorm)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_18 (Conv1D)  │ (None, 128, 128)  │     65,664 │ rms_norm_21[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_19 (Conv1D)  │ (None, 128, 128)  │     65,664 │ rms_norm_21[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_20 (Conv1D)  │ (None, 128, 128)  │     65,664 │ rms_norm_21[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ causal_linear_atte… │ (None, 128, 128)  │          0 │ conv1d_18[0][0],  │
│ (CausalLinearAtten… │                   │            │ conv1d_19[0][0],

 Total params: 1,415,105 (5.40 MB)

 Trainable params: 1,415,105 (5.40 MB)

 Non-trainable params: 0 (0.00 B)

leak test (positions before 100 must not change): 0.0
Epoch 1/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 61s 92ms/step - accuracy: 0.2513 - loss: 2.8086 - val_accuracy: 0.3957 - val_loss: 2.1012 - learning_rate: 0.0010
Epoch 2/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 23s 75ms/step - accuracy: 0.4657 - loss: 1.8236 - val_accuracy: 0.4651 - val_loss: 1.8370 - learning_rate: 0.0010
Epoch 3/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 24s 80ms/step - accuracy: 0.5150 - loss: 1.6228 - val_accuracy: 0.4888 - val_loss: 1.7539 - learning_rate: 0.0010
Epoch 4/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 26s 85ms/step - accuracy: 0.5376 - loss: 1.5293 - val_accuracy: 0.5045 - val_loss: 1.6914 - learning_rate: 0.0010
Epoch 5/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 24s 81ms/step - accuracy: 0.5519 - loss: 1.4723 - val_accuracy: 0.5133 - val_loss: 1.6662 - learning_rate: 0.0010
Epoch 6/30
300/300 ━━━━━━━━━━━━━━━━━━━━ 24s 79ms/step - accuracy: 0.5622 - loss: 1.4310 - val_accuracy: 0.5177 - val_loss: 1.6442 - learning_rate: 0.0010
Epoch 7/30
300/300 ━━━

In [4]:
import os, time, urllib.request
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

# ============================================================
# CONFIG
# ============================================================
SEQ_LEN, EMBED_DIM, NUM_HEADS, FF_DIM, NUM_LAYERS = 128, 128, 4, 512, 4
DROPOUT, BATCH_SIZE, EPOCHS, STEPS_PER_EPOCH = 0.1, 64, 30, 300
SEEDS = [0, 1]
# (name, attention type, conv kernel).  linear K=1 and softmax have the SAME parameter count.
CONFIGS = [("linear K=1 (Dense QKV)", "linear", 1),
           ("linear K=4 (Conv QKV)",  "linear", 4),
           ("softmax (Dense QKV)",    "softmax", 1)]

# ============================================================
# Data (same split as before: last 10% held out; globally shuffled windows)
# ============================================================
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
if not os.path.exists("tinyshakespeare.txt"):
    urllib.request.urlretrieve(url, "tinyshakespeare.txt")
text = open("tinyshakespeare.txt", "r", encoding="utf-8").read()
chars = sorted(set(text)); vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
encoded = np.array([char2idx[c] for c in text], dtype=np.int32)
n_val = int(0.1 * len(encoded))
train_ids, val_ids = encoded[:-n_val], encoded[-n_val:]

def make_ds(data, training):
    data_tf = tf.constant(data, dtype=tf.int32)
    n = len(data) - SEQ_LEN - 1
    def gather(ix):
        pos = ix[:, None] + tf.range(SEQ_LEN + 1)[None, :]
        seq = tf.gather(data_tf, pos)
        return seq[:, :-1], seq[:, 1:]
    if training:
        starts = tf.data.Dataset.from_tensor_slices(np.arange(n, dtype=np.int32))
        starts = starts.shuffle(n, reshuffle_each_iteration=True).repeat()
    else:
        starts = tf.data.Dataset.from_tensor_slices(np.arange(0, n, SEQ_LEN, dtype=np.int32))
    return starts.batch(BATCH_SIZE).map(gather, num_parallel_calls=tf.data.AUTOTUNE).prefetch(2)

# ============================================================
# Layers
# ============================================================
class RMSNorm(layers.Layer):
    def build(self, shape):
        self.w = self.add_weight(shape=(shape[-1],), initializer="ones", name="w")
    def call(self, x):
        return x * tf.math.rsqrt(tf.reduce_mean(tf.square(x), -1, keepdims=True) + 1e-6) * self.w

class PosEmb(layers.Layer):
    def __init__(self, T, d, **kw):
        super().__init__(**kw); self.emb = layers.Embedding(T, d)
    def call(self, x):
        return x + self.emb(tf.range(tf.shape(x)[1]))

class CausalLinearAttention(layers.Layer):
    def __init__(self, T, heads, **kw):
        super().__init__(**kw); self.T, self.h = T, heads
    def call(self, inputs):
        q, k, v = inputs
        B = tf.shape(q)[0]; d = q.shape[-1]; dh = d // self.h
        sp = lambda t: tf.reshape(t, (B, self.T, self.h, dh))
        q = tf.nn.elu(sp(q)) + 1.0; k = tf.nn.elu(sp(k)) + 1.0; v = sp(v)
        S = tf.cumsum(tf.einsum('bthd,bthe->bthde', k, v), axis=1)
        z = tf.cumsum(k, axis=1)
        num = tf.einsum('bthd,bthde->bthe', q, S)
        den = tf.einsum('bthd,bthd->bth', q, z)[..., None] + 1e-6
        return tf.reshape(num / den, (B, self.T, d))

def build_model(kind, conv_k):
    def block(x):
        h = RMSNorm()(x)
        if kind == "linear":
            q = layers.Conv1D(EMBED_DIM, conv_k, padding="causal")(h)
            k = layers.Conv1D(EMBED_DIM, conv_k, padding="causal")(h)
            v = layers.Conv1D(EMBED_DIM, conv_k, padding="causal")(h)
            a = layers.Dense(EMBED_DIM)(CausalLinearAttention(SEQ_LEN, NUM_HEADS)([q, k, v]))
        else:
            a = layers.MultiHeadAttention(NUM_HEADS, EMBED_DIM // NUM_HEADS)(h, h, use_causal_mask=True)
        x = layers.Add()([x, a])
        f = layers.Dense(FF_DIM, activation="silu")(RMSNorm()(x))
        f = layers.Dropout(DROPOUT)(layers.Dense(EMBED_DIM)(f))
        return layers.Add()([x, f])
    I = layers.Input(shape=(SEQ_LEN,), dtype="int32")
    x = PosEmb(SEQ_LEN, EMBED_DIM)(layers.Embedding(vocab_size, EMBED_DIM)(I))
    for _ in range(NUM_LAYERS): x = block(x)
    return keras.Model(I, layers.Dense(vocab_size)(RMSNorm()(x)))

def leak_ok(model):
    a = np.random.randint(0, vocab_size, (1, SEQ_LEN)); b = a.copy()
    b[0, 100] = (b[0, 100] + 1) % vocab_size
    return np.abs(model(a, training=False).numpy()[0, :100] - model(b, training=False).numpy()[0, :100]).max() < 1e-4

# ============================================================
# Run every config x seed
# ============================================================
rows = []
for name, kind, conv_k in CONFIGS:
    for seed in SEEDS:
        keras.backend.clear_session(); keras.utils.set_random_seed(seed)
        model = build_model(kind, conv_k)
        assert leak_ok(model), f"FUTURE LEAK in {name}"
        model.compile(optimizer=keras.optimizers.AdamW(1e-3, weight_decay=0.01, clipnorm=1.0),
                      loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
        t0 = time.time()
        h = model.fit(make_ds(train_ids, True), epochs=EPOCHS, steps_per_epoch=STEPS_PER_EPOCH,
                      validation_data=make_ds(val_ids, False), verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
                                 keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=1, min_lr=1e-5)])
        ep = len(h.history["val_loss"])
        rows.append(dict(config=name, seed=seed, params=model.count_params(),
                         best_val=min(h.history["val_loss"]), epochs=ep, sec_per_epoch=(time.time() - t0) / ep))
        print(f"{name:26s} seed={seed} params={rows[-1]['params']:,} best_val={rows[-1]['best_val']:.4f} "
              f"epochs={ep} {rows[-1]['sec_per_epoch']:.0f}s/epoch", flush=True)
        pd.DataFrame(rows).to_csv("compare_results.csv", index=False)        # saved after every run

df = pd.DataFrame(rows)
t = df.groupby("config", sort=False).agg(params=("params", "first"), n=("seed", "count"),
        val_loss=("best_val", "mean"), std=("best_val", "std"), sec_per_epoch=("sec_per_epoch", "mean")).round(4)
print("\n", t)
print("\nreference (earlier 462K-param runs, same split): linear 1.634, softmax 1.622")


linear K=1 (Dense QKV)     seed=0 params=825,281 best_val=1.5496 epochs=22 22s/epoch
linear K=1 (Dense QKV)     seed=1 params=825,281 best_val=1.5467 epochs=24 22s/epoch
linear K=4 (Conv QKV)      seed=0 params=1,415,105 best_val=1.5602 epochs=23 25s/epoch
linear K=4 (Conv QKV)      seed=1 params=1,415,105 best_val=1.5742 epochs=22 25s/epoch
softmax (Dense QKV)        seed=0 params=825,281 best_val=1.5281 epochs=23 10s/epoch
softmax (Dense QKV)        seed=1 params=825,281 best_val=1.5297 epochs=24 10s/epoch

                          params  n  val_loss     std  sec_per_epoch
config                                                             
linear K=1 (Dense QKV)   825281  2    1.5482  0.0020        22.1899
linear K=4 (Conv QKV)   1415105  2    1.5672  0.0099        24.7677
softmax (Dense QKV)      825281  2    1.5289  0.0012        10.0254

reference (earlier 462K-param runs, same split): linear 1.634, softmax 1.622
